# Netflix RecSys — Model Development & Comparison
SVD vs User-Based CF | RMSE + MAP@10

In [8]:
# !pip install scikit-surprise

In [9]:
import pandas as pd
import numpy as np
from surprise import Dataset, Reader, SVD, KNNBasic
from surprise.model_selection import train_test_split
from surprise import accuracy
from collections import defaultdict

print('Libraries loaded ✅')

Libraries loaded ✅


## 1. Load Data

In [10]:
df     = pd.read_csv('data/netflix_clean.csv', parse_dates=['date'])
movies = pd.read_csv('data/movies_clean.csv')

print(f'Ratings : {len(df):,}')
print(f'Users   : {df.user_id.nunique():,}')
print(f'Movies  : {df.movie_id.nunique():,}')
df.head()

Ratings : 3,778,410
Users   : 10,000
Movies  : 4,490


,movie_id,user_id,rating,date,rating_year,rating_month
0,1,1488844,3,2005-09-06,2005,9
1,1,30878,4,2005-12-26,2005,12
2,1,1248029,3,2004-04-22,2004,4
3,1,1080361,3,2005-03-28,2005,3
4,1,558634,4,2004-12-14,2004,12


## 2. Train / Test Split (time-based)
Train on ratings before 2005-10-01, test on after. Keeps temporal integrity.

In [11]:
SPLIT_DATE = '2005-10-01'

train_df = df[df['date'] <  SPLIT_DATE].copy()
test_df  = df[df['date'] >= SPLIT_DATE].copy()

# Keep only test users/movies that exist in train
train_users  = set(train_df['user_id'])
train_movies = set(train_df['movie_id'])
test_df = test_df[
    test_df['user_id'].isin(train_users) &
    test_df['movie_id'].isin(train_movies)
].copy()

print(f'Train : {len(train_df):,}  ({train_df.date.min().date()} → {train_df.date.max().date()})')
print(f'Test  : {len(test_df):,}  ({test_df.date.min().date()} → {test_df.date.max().date()})')

Train : 3,527,509  (1999-12-21 → 2005-09-30)
Test  : 209,123  (2005-10-01 → 2005-12-31)


## 3. Build Surprise Datasets

In [12]:
reader = Reader(rating_scale=(1, 5))

train_data = Dataset.load_from_df(
    train_df[['user_id', 'movie_id', 'rating']], reader
).build_full_trainset()

# Testset from test_df
testset = [
    (row.user_id, row.movie_id, row.rating)
    for row in test_df[['user_id', 'movie_id', 'rating']].itertuples()
]

print(f'Trainset size : {train_data.n_ratings:,}')
print(f'Testset size  : {len(testset):,}')

Trainset size : 3,527,509
Testset size  : 209,123


## 4. Model 1 — SVD (Matrix Factorization)

In [13]:
print('Training SVD...')
svd = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd.fit(train_data)

svd_preds = svd.test(testset)
svd_rmse  = accuracy.rmse(svd_preds, verbose=False)
svd_mae   = accuracy.mae(svd_preds,  verbose=False)

print(f'SVD  RMSE : {svd_rmse:.4f}')
print(f'SVD  MAE  : {svd_mae:.4f}')

Training SVD...
SVD  RMSE : 0.8885
SVD  MAE  : 0.6775


## Model Justification

### Why SVD (Matrix Factorization)?

The Netflix Prize dataset has **91.58% sparsity**, meaning most user-movie pairs have no rating at all. SVD handles this well because instead of relying on direct user-item overlap, it decomposes the rating matrix into latent factors — hidden representations that capture *why* a user likes something (e.g. preference for a genre, era, or style) without those features being explicitly present in the data.

**Key reasons for choosing SVD:**
- Learns global patterns across all users and movies simultaneously
- Handles sparse data far better than neighborhood-based methods
- Trains in ~1 minute on 3.7M ratings — computationally efficient
- Matrix Factorization approaches won the original Netflix Prize competition, validating its suitability for this exact problem



## 5. Model 2 — Item-Based CF (KNN)

In [15]:
print('Training Item-Based CF...')
ibcf = KNNBasic(
    k=40, min_k=5,
    sim_options={'name': 'cosine', 'user_based': False},
    verbose=False
)
ibcf.fit(train_data)

ibcf_preds = ibcf.test(testset)
ibcf_rmse  = accuracy.rmse(ibcf_preds, verbose=False)
ibcf_mae   = accuracy.mae(ibcf_preds,  verbose=False)

print(f'IBCF RMSE : {ibcf_rmse:.4f}')
print(f'IBCF MAE  : {ibcf_mae:.4f}')

Training Item-Based CF...
IBCF RMSE : 1.0002
IBCF MAE  : 0.7730


##  Why Not Other Models?

### Model Choice Justification

| Model | Why Not Chosen |
|---|---|
| User-Based CF | Needs users with overlapping ratings. At 91% sparsity, most user pairs share very few movies — similarity scores become unreliable |
| Neural CF | More expressive but requires significant tuning, longer training, and harder to interpret. SVD already delivers strong results with less complexity |
| Content-Based | Movie metadata (title + year only) is too thin to build meaningful item profiles |
| Item-Based CF | Chosen as the comparison baseline — fewer unique movies (4,490) than users (10,000) makes similarity computation faster, but still suffers from sparsity |

A simpler, well-understood model with strong performance is preferred over unnecessary complexity.


## 6. MAP@10
Relevant = actual rating ≥ 3.5 (as specified in problem statement).
For each test user, score all their test movies with the model, rank, take top-10.

In [16]:
RELEVANCE_THRESHOLD = 3.5

def compute_map_at_k(predictions, k=10, threshold=RELEVANCE_THRESHOLD):
    """Compute MAP@K from surprise predictions list."""

    # Group predictions by user
    user_preds = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_preds[uid].append((est, true_r))

    ap_list = []

    for uid, preds in user_preds.items():
        # Only score users that have at least one relevant item
        relevant_total = sum(1 for _, true_r in preds if true_r >= threshold)
        if relevant_total == 0:
            continue

        # Sort by estimated rating descending → top-K
        preds_sorted = sorted(preds, key=lambda x: x[0], reverse=True)[:k]

        hits = 0
        precision_sum = 0.0

        for rank, (est, true_r) in enumerate(preds_sorted, start=1):
            if true_r >= threshold:
                hits += 1
                precision_sum += hits / rank

        ap = precision_sum / min(relevant_total, k)
        ap_list.append(ap)

    return np.mean(ap_list) if ap_list else 0.0


svd_map  = compute_map_at_k(svd_preds,  k=10)
ibcf_map = compute_map_at_k(ibcf_preds, k=10)

print(f'SVD  MAP@10 : {svd_map:.4f}')
print(f'ibcf MAP@10 : {ibcf_map:.4f}')

SVD  MAP@10 : 0.7767
ibcf MAP@10 : 0.7014


## 7. Comparison Table

In [17]:
results = pd.DataFrame({
    'Model'  : ['SVD (MF)', 'Item-Based CF'],
    'RMSE'   : [svd_rmse,  ibcf_rmse],
    'MAE'    : [svd_mae,   ibcf_mae],
    'MAP@10' : [svd_map,   ibcf_map],
})

results = results.set_index('Model').round(4)
print(results.to_string())
results

                 RMSE     MAE  MAP@10
Model                                
SVD (MF)       0.8885  0.6775  0.7767
Item-Based CF  1.0002  0.7730  0.7014


,RMSE,MAE,MAP@10
Model,,,
SVD (MF),0.8885,0.6775,0.7767
Item-Based CF,1.0002,0.7730,0.7014


##  Metric Analysis

### Why the Metrics Look the Way They Do

**RMSE — SVD (0.8885) vs IBCF (1.0002)**

SVD learns the full rating matrix globally, capturing subtle preference patterns across all users simultaneously. IBCF only looks at the 40 most similar items for each prediction — with high sparsity, those neighbors are often weak matches, causing predictions to fall back toward the global mean more aggressively. SVD's lower RMSE means its rating predictions are on average within 0.89 stars of the true rating.

**MAP@10 — SVD (0.7767) vs IBCF (0.7014)**

MAP@10 measures whether the top-10 recommendations are actually relevant (rating ≥ 3.5). SVD's latent factors capture the *strength* of preference, not just similarity — so it ranks genuinely liked movies higher. IBCF treats all similar items with roughly equal weight, which hurts ranking quality. SVD's MAP@10 of 0.7767 means on average ~78% of the top-10 recommendations are relevant to the user.

**RMSE vs MAP@10 — The Tradeoff**

These two metrics measure different things:
- **RMSE** = how accurately the model predicts the exact star rating
- **MAP@10** = how well the model ranks relevant items at the top of recommendations

A model can have low RMSE but poor MAP@10 if it predicts ratings accurately but still ranks irrelevant items above relevant ones. SVD wins on both metrics, making it the stronger practical choice for a real recommendation system where ranking quality matters more than exact rating prediction.


## 8. Top-10 Recommendations — Sample Users

In [18]:
def get_top_k_recs(model, user_id, train_df, movies_df, k=10):
    """Return top-K unseen movies for a user using the given model."""
    seen = set(train_df[train_df['user_id'] == user_id]['movie_id'])
    all_movies = movies_df['movie_id'].unique()
    unseen = [m for m in all_movies if m not in seen]

    preds = [(m, model.predict(user_id, m).est) for m in unseen]
    preds.sort(key=lambda x: x[1], reverse=True)
    top_k = preds[:k]

    recs = pd.DataFrame(top_k, columns=['movie_id', 'predicted_rating'])
    recs = recs.merge(movies_df[['movie_id', 'title', 'year']], on='movie_id')
    return recs[['title', 'year', 'predicted_rating']].round(3)


# Pick 3 sample users from test set
sample_users = test_df['user_id'].value_counts().head(3).index.tolist()

for uid in sample_users:
    print(f'\n--- User {uid} | SVD Top-10 ---')
    print(get_top_k_recs(svd, uid, train_df, movies).to_string(index=False))


--- User 2118461 | SVD Top-10 ---
                                                        title   year  predicted_rating
         Inu-Yasha: The Movie 3: Swords of an Honorable Ruler 2002.0             5.000
                                               Lost: Season 1 2004.0             5.000
Frosty's Winter Wonderland / 'Twas the Night Before Christmas 1974.0             5.000
                                              Alias: Season 4 2005.0             4.995
                My Fair Lady: Special Edition: Bonus Material 1964.0             4.925
                        Little House on the Prairie: Season 3 1976.0             4.914
                        The Wizard of Oz: Collector's Edition 1939.0             4.902
                                 Mary Poppins: Bonus Material 1964.0             4.901
                                Santa Claus Is Comin' to Town 1970.0             4.896
                                   An Officer and a Gentleman 1982.0             4.882

--- Use

## 9. Success & Failure Cases

In [19]:
# Predictions as DataFrame for easy analysis
svd_pred_df = pd.DataFrame([
    {'user_id': uid, 'movie_id': iid, 'true_r': tr, 'est_r': est}
    for uid, iid, tr, est, _ in svd_preds
])
svd_pred_df['error'] = abs(svd_pred_df['true_r'] - svd_pred_df['est_r'])
svd_pred_df = svd_pred_df.merge(movies[['movie_id', 'title']], on='movie_id', how='left')

print('=== Success Cases (error < 0.2) ===')
print(svd_pred_df[svd_pred_df['error'] < 0.2]
      [['title','true_r','est_r','error']]
      .sample(5, random_state=1).to_string(index=False))

print('\n=== Failure Cases (error > 2.0) ===')
print(svd_pred_df[svd_pred_df['error'] > 2.0]
      [['title','true_r','est_r','error']]
      .sample(5, random_state=1).to_string(index=False))

=== Success Cases (error < 0.2) ===
                           title  true_r    est_r    error
          The Godfather, Part II       5 5.000000 0.000000
The Blues Brothers: Extended Cut       5 5.000000 0.000000
                       Tank Girl       3 2.912490 0.087510
                           Speed       3 3.114479 0.114479
                   Random Hearts       4 3.838895 0.161105

=== Failure Cases (error > 2.0) ===
                                      title  true_r    est_r    error
                                  Bedazzled       1 3.261036 2.261036
                               Nick of Time       1 4.340844 3.340844
                                  The Quest       4 1.915215 2.084785
House of Cards Trilogy II: To Play the King       1 4.214299 3.214299
                                Man on Fire       1 3.001300 2.001300


## Failure Analysis

### Why Do Failure Cases Occur?

The failure cases show a consistent pattern: movies the user rated **1 star** are predicted as **3.0–4.3 stars**. This is a known problem called **popularity bias** — the model is biased toward the global mean rating (3.385).

**Root causes:**
1. **Class imbalance** — 1-star ratings are rare in the dataset. The model has seen far fewer examples of extreme dislikes, so it defaults to predicting near the average
2. **Cold signal for dislikes** — users who strongly dislike a movie often share few other ratings with people who also dislike it, making it hard to learn that pattern
3. **Latent factors capture popularity** — SVD latent factors tend to encode what most users like, not what specific users hate

**Business implication:** The system is safer to use for discovering content users will enjoy, but should not be relied upon to filter out content users will strongly dislike. A hybrid approach incorporating explicit dislike signals would improve this.


## 10. Save Models & Results

In [20]:
import pickle, os
os.makedirs('models', exist_ok=True)

with open('models/svd.pkl',  'wb') as f: pickle.dump(svd,  f)
with open('models/ibcf.pkl', 'wb') as f: pickle.dump(ibcf, f)

results.to_csv('data/model_results.csv')
print('Models + results saved ✅')

Models + results saved ✅


## Cold Start Discussion

Cold start is one of the most fundamental challenges in collaborative filtering systems.
It refers to situations where the model has insufficient interaction history to make
reliable recommendations. There are three distinct cold start scenarios:

---

### 1. New User Problem
A user with zero or very few ratings provides no signal for collaborative filtering.

**What happens in our system:**
- SVD cannot compute a meaningful latent vector for a new user
- The model defaults to `global mean (3.385) + item bias` — producing the same
  generic popular recommendations for every new user
- IBCF cannot find similar users because there is no rating history to compare

**Evidence in our results:**
- Users with fewer than 5 ratings in our training set were filtered out during
  cleaning — meaning our evaluation metrics are optimistic and do not reflect
  cold-start performance
- In production, these users would receive popularity-based fallbacks, not
  personalised recommendations

**Mitigation strategies:**
- **Onboarding survey** — ask new users to rate 5–10 movies they have already
  seen to bootstrap their preference profile
- **Demographic-based recommendations** — use age, region, or device type as
  proxy signals before ratings are available
- **Popularity fallback** — recommend globally top-rated content until sufficient
  ratings are collected (typically 10–20 ratings)
- **SVD++ implicit feedback** — treat any interaction (click, browse, partial
  watch) as a weak signal even without an explicit rating

---

### 2. New Item Problem
A movie that has received no ratings has no co-occurrence signal for IBCF and
no item bias term learned by SVD.

**What happens in our system:**
- IBCF cannot compute cosine similarity for an unrated item — it simply cannot
  appear in any recommendation list
- SVD assigns the item zero bias, meaning it will only be recommended based on
  the global mean, making it nearly invisible to the ranking

**Evidence in our results:**
- Our dataset covers 4,383 movies after filtering. Movies with fewer than 10
  ratings were removed — these are exactly the items most affected by cold start
- New releases added after the training window (post Oct 2005) would receive
  no recommendations at all

**Mitigation strategies:**
- **Content-based features** — use genre, director, cast, or release year to
  find similar items to already-rated movies, bypassing the need for rating
  co-occurrence
- **Hybrid model** — combine SVD predictions with a content-based similarity
  score: `final_score = α × SVD_score + (1-α) × content_score`
- **Item metadata embeddings** — encode genre and year as features and
  initialise the item latent vector from these rather than from zero

---

### 3. Sparse User Problem
Even users who have rated some movies may have such sparse histories that
collaborative filtering produces unreliable results.

**What happens in our system:**
- IBCF requires `min_k=5` neighbours with co-rated items. For sparse users,
  this threshold is often not met, and the model falls back to the global mean
- SVD handles sparsity better through regularisation, but users with very few
  ratings will have poorly estimated bias terms

**Evidence in our results:**
- The failure cases shown above (1-star movies predicted at 3.0–4.3) are
  largely sparse-user artefacts — the model has seen too few negative signals
  from that user to override the popularity bias

**Mitigation strategies:**
- **Lower the min_k threshold** for sparse users and compensate with stronger
  regularisation
- **Matrix completion techniques** — use side information to impute missing
  ratings before training
- **Session-based recommendations** — for users with very sparse long-term
  history, use short-term session context (last 3–5 interactions) as the
  primary signal

---

### Summary Table

| Scenario | Impact on SVD | Impact on IBCF | Recommended Fix |
|---|---|---|---|
| New user (0 ratings) | Global mean only | Cannot run | Onboarding survey + popularity fallback |
| New item (0 ratings) | Zero bias term | Invisible to ranking | Content-based hybrid |
| Sparse user (<10 ratings) | Poor bias estimate | Neighbour threshold fails | Implicit feedback + lower min_k |

---

### Business Implication
Cold start directly affects the first-session experience — the most critical
moment for user retention. A user who receives irrelevant recommendations on
their first visit is unlikely to return. Addressing cold start is therefore not
just a modelling concern but a product priority. A practical production system
would implement a tiered recommendation strategy: content-based or
popularity-based recommendations for new users, transitioning to collaborative
filtering once sufficient interaction history is accumulated.